# Langchain 2026
Doby, kdy obsah tohoto repozitáře sloužil jako podklad pro workshopy, nenávratně zmizely. Nicméně čas od času se mi toto místo hodí coby skladiště poznámek. Navíc popis nějakého tématu člověku pomáhá utřídit myšlenky. Proč o tom v tomto odstavci píšu? Abych se mohl vyvinit, pokud by některá část povídání byla napsána příliš nesrozumitelně. A navíc u osobních poznámek člověk může s klidem prohlást "Nemám absolutní tušení, jak tohleto funguje".  

Laskavý čtenář si možná všimne, že ve stejném adresáři jako tento notebook bydlí i jiný spisek věnovaný Langchainu. Nicméně ten byl updatován naposled před dvěma lety. Během tohoto času byl Lanchain (opět) dramaticky překopán. Mohl bych sice upravovat původní odkument, ale byla by s tím zbytečně velká práce. Navíc porovnání toho, jak dosáhnout cíle X nyní a kdysi má též své kouzlo...

In [24]:
import json
import os
import re
from dotenv import load_dotenv
from langchain.messages import AIMessage, SystemMessage, HumanMessage
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import RemoveMessage
from langchain.agents.middleware import after_model
from langchain.agents import AgentState
from langgraph.runtime import Runtime
from langchain.agents.middleware import SummarizationMiddleware
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, SparseVectorParams
from qdrant_client import QdrantClient, models
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode


from langchain_text_splitters import RecursiveCharacterTextSplitter

from pydantic import BaseModel, Field

load_dotenv()

True

### Prerekvizity
V následujícím textu budeme používat jazykové modely bydlící na serverech patřících OpenAI. Abychom se k nim připojili, potřebujeme API klíč. Bylo by nicméně poněkud nebezpečné psát ho přímo do notebooku - člověk by ho tu mohl zapomenout a tak by se onen údaj mohl dostat na Github. To by ve výsledku mohlo vést k průvanu v peněžence. Proto si takovéto citlivé věci uložíme do souboru ".env" - ten vytvoříme ve stejném adresáři, ve kterém se nalézá tento notebook. API klíč (a obodobě případné další citlivé informace) do něj vložíme ve formátu
```
OPENAI_AI_KEY="API_KLIC"
```
Do proměnných prostředí se potom tyto údaje dostanou s pomocí balíčku [python-dotenv](https://pypi.org/project/python-dotenv/), přesněji tedy níže uvedeným kódem. V rámci Pythonu se k nim dostaneme příkazy ala *os.getenv("OPENAI_AI_KEY")*

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

Pro samotný Langchain nebude stačit pouze ["pip install langchain"](https://pypi.org/project/langchain/), budeme muset udělat i ["pip install langchain-openai"](https://pypi.org/project/langchain-openai/). Tento balíček by snad měl stačit i pokud bychom používali OpenAI modely z Azure, tj. snad by v takovém případě nebylo nutné instalovat i ["pip install langchain-azure-ai"](https://pypi.org/project/langchain-azure-ai/).  
Pokud bychom při instalaci dostali hlášku "pip._vendor.packaging.version.InvalidVersion: Invalid version: 'hosting'", tak máme asi zastaralý pip a musíme ho upgradovat ("python -m pip install --upgrade pip").  

### Modely
#### Invoke
Nejjednodušší komunikaci s LLM modelem si ukážeme na následujícím příkladu. 

In [3]:
import os
from langchain.chat_models import init_chat_model

llm_model = init_chat_model(
    model="gpt-4.1-mini",
    model_provider="openai",
    api_key=os.getenv("OPENAI_API_KEY")
)
model_response = llm_model.invoke("Ahoj, jak se máš?")
print(model_response.content)

Ahoj! Mám se dobře, děkuji za optání. Jak se máš ty? Můžu ti s něčím pomoci?


Jedná se opravdu o ten nejprimitivnější příklad, který lze ukázat. Defaultně není v objektu reprezentujícím jazykový model ani přítomná paměť. 

In [8]:
model_response_2 = llm_model.invoke("Na co jsem se tě před chvílí ptal?")
print(model_response_2.content)

Promiň, ale nemám přístup k předchozím konverzacím nebo dotazům, které jsi mi položil před chvílí. Můžeš mi prosím připomenout, na co ses ptal? Rád ti pomohu!


V případě, že bychom pracovali s Azurem, vypadal by kód nějak takto:
```python
import os
from langchain_openai import AzureChatOpenAI

llm_model = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    azure_deployment="NAME_OF_MODEL_DEPLOYMENT",
    api_version="API_VERSION_FOUND_IN_FOUNDRY",
    api_key=os.getenv("AZURE_OPENAI_API_KEY")                 
)

model_response = llm_model.invoke("Ahoj, jak se máš?")
print(model_response.content)
```

Pokud bychom chtěli do *invoke* metody vložit něco složitějšího (systémový prompt, předchozí kola konverzace), použijeme objekty SystemMessage, HumanMessage a AIMessage vložené do listu.

In [9]:
from langchain.messages import AIMessage, SystemMessage, HumanMessage

messages_list = [
    SystemMessage("Jsi králík"),
    HumanMessage("Ahoj"),
    AIMessage("Ahoj *chroust chroust*"),
    HumanMessage("Jaká zelenina je nejlepší?")
]

model_response = llm_model.invoke(messages_list)
print(model_response.content)

Jé, moje oblíbená je určitě mrkev! Je sladká a pěkně křupavá. Ale taky mám rád třeba salát nebo pampelišku. Co ty, máš rád nějakou zeleninu?


Alternativně můžeme použít i slovník.

In [10]:
messages_list = [
    {"role": "system", "content":"Jsi králík"},
    {"role": "user", "content":"Ahoj"},
    {"role": "assistant", "content":"Ahoj *chroust chroust*"},
    {"role": "user", "content":"Jaká zelenina je nejlepší?"}
]
model_response = llm_model.invoke(messages_list)
print(model_response.content)

Pro králíky jsou nejlepší čerstvá mrkev, listový salát, petržel a celer — to všechno je výborná a zdravá zelenina pro nás! Máš nějakou oblíbenou? *čmuch čmuch*


V rámci objektu, který nám *invoke* metoda vrátí, nalezneme i spotřebu tokenů.

In [11]:
model_response.usage_metadata

{'input_tokens': 45,
 'output_tokens': 54,
 'total_tokens': 99,
 'input_token_details': {'audio': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 0}}

#### Streamování
Pakliže to model podporuje, nemusí se na odpověď čekat, dokud není celá vyrobená, ale mohou se zobrazovat její části přicházející ve streamu. Tehdy je tedy potřeba namísto metody *invoke* použít metodu *stream*.

In [13]:
for one_chunk in llm_model.stream("Jak by vypadal svět bez mrkve? Odpověz maximálně jedním odstavcem."):
    print(one_chunk.text, end="|", flush=True)

|Sv|ět| bez| mr|k|ve| by| post|rá|dal| jednu| z| nej|roz|ší|řen|ější|ch| a| vý|živ|ově| hodnot|ných| zelen|in|,| což| by| ov|liv|n|ilo| jak| lids|kou| str|avu|,| tak| i| zem|ě|d|ěl|ství| a| kuch|yni| po| cel|ém| svět|ě|;| ch|yb|ě|ly| by| nám| nejen| její| slad|ká| chu|ť| a| bo|hat|ý| zd|roj| beta|-kar|ot|enu|,| ale| i| význam|né| využ|ití| v| tradi|čních| rece|p|tech|,| dě|ts|ké| vý|živ|ě| a| kr|men|í| domác|ích| z|ví|ř|at|,| což| by| moh|lo| vé|st| k| hled|ání| alternativ| a| mož|ná| i| ke| změ|n|ám| v| kulturn|ích| z|vy|kl|ost|ech| a| gastronom|ii|.||||

#### Tooly
Pro vložení toolů (alias funkcí, které model může použít, pokud to uzná za vhodné) do modelu musíme před danou funkci vložit dekorátor *tool*. Následně je třeba daný tool přilepit k modelu skrz jeho metodu *bind_tools*. Pozor - metoda, ze které vyrábíme tool, musí mít docstring - bez něj Langchain nahlásí chybu. To proto, že právě docstring se používá při vyhodnocování, zda metodu jako tool použít.  
Když posléze model s toolem provoláme invokem, vidíme, že nic nevidíme.

In [14]:
from langchain.tools import tool

@tool
def get_number_of_vegetables(vegetable_name:str)->str:
    """Get number of available pieces of vegetable."""
    number_of_vegetables = 42
    return f"We have {number_of_vegetables} of {vegetable_name}."

llm_with_tools = llm_model.bind_tools([get_number_of_vegetables])

model_response = llm_with_tools.invoke("Kolik máme ve skladu mrkví?")
print(model_response.content)

Když se totiž tímto způsobem tooly použijí, je na uživateli, aby tool provolal a výsledek následně znova vložil do modelu. Pokud chceme, aby takovéto akce prováděl stroj sám od sebe, musíme použít agenty (o tom ale až dále). Nyní se bez jejich použití zkusme dostat z jámy, kterou jsme si sami vykopali.  
Napřed si vytvořme list, ve kterém bude historie konverzace. Bacha - nesmíme zapomenout na iniciální (byť pro nás prázdnou) prvotní odpověď stroje.

In [15]:
messages_list = [
    HumanMessage("Kolik máme ve skladu mrkví?"),
    model_response
]

Podívejme se, co vlastně model z hlediska toolů vrátil. Vidíme, že správně vypreparoval parametr pro volání funkce.

In [16]:
for one_tool_call in model_response.tool_calls:
    print(f"Tool: {one_tool_call["name"]}")
    print(f"Tool: {one_tool_call["args"]}")

Tool: get_number_of_vegetables
Tool: {'vegetable_name': 'mrkev'}


Nyní tedy naši *tool* metodu provoláme. Všimněme si, že to provádíme skrz metodu *invoke*, kterou naše funkce získala díky dekorátoru. Výsledek opět vložíme do listu s historií konverzace.

In [17]:
for one_tool_call in model_response.tool_calls:
    tool_result = get_number_of_vegetables.invoke(one_tool_call)
    messages_list.append(tool_result)

Všimněme si, že "toolí" příspěvek má svůj vlastní typ a že AI message je poměrně obsáhlá.

In [18]:
messages_list

[HumanMessage(content='Kolik máme ve skladu mrkví?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 59, 'total_tokens': 80, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b51326e9dc', 'id': 'chatcmpl-E0t8Z6GaxNJoD5ANkr3rtXzQBCwui', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f5792-abbc-7fa3-8996-b426ae3edd29-0', tool_calls=[{'name': 'get_number_of_vegetables', 'args': {'vegetable_name': 'mrkev'}, 'id': 'call_BXJdrbG1BZXIcXqqoNNpWBvT', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 59, 'output_tokens

Nakonec do metody *invoke* navázané na model s tooly vložíme historii konverzace a získáme tak finální odpověď.

In [19]:
final_response = llm_with_tools.invoke(messages_list)
final_response.text

'Ve skladu máme 42 kusů mrkví.'

Pro přehlednost vložme celý kód do jedné jupyteří buňky:

In [20]:
from langchain.tools import tool

@tool
def get_number_of_vegetables(vegetable_name:str)->str:
    """Get number of available pieces of vegetable."""
    number_of_vegetables = 42
    return f"We have {number_of_vegetables} of {vegetable_name}."

llm_with_tools = llm_model.bind_tools([get_number_of_vegetables])

user_question = "Kolik máme ve skladu mrkví?"
model_response = llm_with_tools.invoke(user_question)
messages_list = [user_question, model_response]

for one_tool_call in model_response.tool_calls:
    tool_result = get_number_of_vegetables.invoke(one_tool_call)
    messages_list.append(tool_result)

final_response = llm_with_tools.invoke(messages_list)
final_response.text

'Ve skladu máme 42 kusů mrkve.'

Defaultně není použití toolů pro model povinné.

In [4]:
@tool
def get_number_of_vegetables(vegetable_name:str)->str:
    """Get number of available pieces of vegetable."""
    number_of_vegetables = 42
    return f"We have {number_of_vegetables} of {vegetable_name}."

llm_with_tools = llm_model.bind_tools([get_number_of_vegetables])

model_response = llm_with_tools.invoke("Kolik je planet ve Sluneční soustavě?")

for one_tool_call in model_response.tool_calls:
    print(f"Tool: {one_tool_call["name"]}")
    print(f"Tool: {one_tool_call["args"]}")

Nicméně použití alespoň jednoho toolu můžeme vynutit, pokud do metody *bind_tools* přidáme tool_choice="any". Výsledek ale pak bude podle toho vypadat.

In [6]:
@tool
def get_number_of_vegetables(vegetable_name:str)->str:
    """Get number of available pieces of vegetable."""
    number_of_vegetables = 42
    return f"We have {number_of_vegetables} of {vegetable_name}."

@tool
def get_fluffy_animal()->str:
    """Get number of available pieces of vegetable."""
    return "Králík"

llm_with_tools = llm_model.bind_tools([get_number_of_vegetables, get_fluffy_animal], tool_choice="any")

model_response = llm_with_tools.invoke("Kolik je planet ve Sluneční soustavě?")

for one_tool_call in model_response.tool_calls:
    print(f"Tool: {one_tool_call["name"]}")
    print(f"Tool: {one_tool_call["args"]}")

Tool: get_fluffy_animal
Tool: {}


I když by se důvod pro to hledal docela obtížně, můžeme vynutit použití konkrétního toolu tím, že do parametru *tool_choice* vložíme jméno toolu/funkce jako string.

In [7]:
@tool
def get_number_of_vegetables(vegetable_name:str)->str:
    """Get number of available pieces of vegetable."""
    number_of_vegetables = 42
    return f"We have {number_of_vegetables} of {vegetable_name}."

@tool
def get_fluffy_animal()->str:
    """Get number of available pieces of vegetable."""
    return "Králík"

llm_with_tools = llm_model.bind_tools([get_number_of_vegetables, get_fluffy_animal], tool_choice="get_number_of_vegetables")

model_response = llm_with_tools.invoke("Kolik je planet ve Sluneční soustavě?")

for one_tool_call in model_response.tool_calls:
    print(f"Tool: {one_tool_call["name"]}")
    print(f"Tool: {one_tool_call["args"]}")

Tool: get_number_of_vegetables
Tool: {'vegetable_name': 'planet'}


Pokud bychom chtěli provolat paralelně více toolů (případně paralelně stejný tool víckrát), nemusíme dělat vůbec nic navíc - Langchain toto vykoná defaultně sám od sebe.

In [8]:
@tool
def get_number_of_vegetables(vegetable_name:str)->str:
    """Get number of available pieces of vegetable."""
    number_of_vegetables = 42
    return f"We have {number_of_vegetables} of {vegetable_name}."

@tool
def get_fluffy_animal()->str:
    """Get number of available pieces of vegetable."""
    return "Králík"

llm_with_tools = llm_model.bind_tools([get_number_of_vegetables, get_fluffy_animal])

model_response = llm_with_tools.invoke("Kolik máme ve skladu mrkví a květáků?")

for one_tool_call in model_response.tool_calls:
    print(f"Tool: {one_tool_call["name"]}")
    print(f"Tool: {one_tool_call["args"]}")

Tool: get_number_of_vegetables
Tool: {'vegetable_name': 'mrkev'}
Tool: get_number_of_vegetables
Tool: {'vegetable_name': 'květák'}


#### Formát výstupu
V případě, že potřebujeme vynucovat formát výstupu, použijeme namísto metody *invoke* metodu *with_structured_output*.

In [9]:
import json

json_schema = {
    "title": "Animal",
    "description": "Info about animal",
    "type": "object",
    "properties": {
        "english_name": {
            "type": "string",
            "description": "Animal name in English"
        },
        "number_of_limbs": {
            "type": "integer",
            "description": "Number of limbs (hands, legs, paws, etc.) the animal has"
        },
        "czech_name": {
            "type": "string",
            "description": "Animal name in Czech"
        },
        "first_occurence": {
            "type": "string",
            "description": "When the animal appeared on Earth"
        },
    }
}

model_with_structure = llm_model.with_structured_output(
    json_schema,
    method="json_schema"
)

model_response = model_with_structure.invoke("Co víš o králících?")
model_response

{'english_name': 'Rabbit',
 'czech_name': 'Králík',
 'number_of_limbs': 4,
 'first_occurence': 'Rabbits first appeared approximately 40 million years ago during the Eocene epoch.'}

In [10]:
from pydantic import BaseModel, Field

class Animal(BaseModel):
    """Info about certain animal"""
    english_name: str = Field(description="Animal name in English")
    number_of_limbs: int = Field(description="Number of limbs (hands, legs, paws, etc.) the animal has")
    czech_name: str = Field(description="Animal name in Czech")
    first_occurence: str = Field(description="When the animal appeared on Earth")

model_with_structure = llm_model.with_structured_output(Animal)

model_response = model_with_structure.invoke("Co víš o králících?")
model_response

Animal(english_name='Rabbit', number_of_limbs=4, czech_name='Králík', first_occurence='Approximately 40 million years ago')

## Agenti
#### Tooly
Výše jsme viděli, že práce s tooly s použití samotných modelů (ve smyslu langchainových objektů) je poněkud nepohodlná. Řešení tohoto problému spočívá v použití agentů. U nich se totiž použití toolů odehraje automaticky.  
Všimněme si v následujícím příkladu, že vstupem do *invoke* metody je slovník. Pokud bychom se pokusili předat string, dostali bychom error "InvalidUpdateError: Expected dict, got Kolik máme ve skladu mrkví?".

In [12]:
from langchain.agents import create_agent

@tool
def get_number_of_vegetables(vegetable_name:str)->str:
    """Get number of available pieces of vegetable."""
    number_of_vegetables = 42
    return f"We have {number_of_vegetables} of {vegetable_name}."

llm_model = init_chat_model(
    model="gpt-4.1-mini",
    model_provider="openai",
    api_key=os.getenv("OPENAI_API_KEY")
)

llm_agent = create_agent(
    model=llm_model,
    tools=[get_number_of_vegetables]
)

user_question = "Kolik máme ve skladu mrkví?"
model_response = llm_agent.invoke(
    {"messages": [HumanMessage(user_question)]}
)

model_response["messages"][-1].content

'Ve skladu máme 42 kusů mrkve.'

Pokud bychom chtěli vidět, co se dělo pod kapotou, podíváme se na všechny "messages", nikoli jen na to poslední:

In [13]:
for one_message in model_response["messages"]:
    one_message.pretty_print()

================================ Human Message =================================

Kolik máme ve skladu mrkví?
================================== Ai Message ==================================
Tool Calls:
  get_number_of_vegetables (call_J2Tl7TQn7iPLfQ2LhoEZGRES)
 Call ID: call_J2Tl7TQn7iPLfQ2LhoEZGRES
  Args:
    vegetable_name: mrkev
================================= Tool Message =================================
Name: get_number_of_vegetables

We have 42 of mrkev.
================================== Ai Message ==================================

Ve skladu máme 42 kusů mrkve.


#### Systémový prompt
Systémový prompt píšeme přímo do funkce agenta vytvářející:

In [14]:
llm_agent = create_agent(
    model=llm_model,
    system_prompt="Jsi králík"
)

user_question = "Kdo jsi?"
model_response = llm_agent.invoke(
    {"messages": [HumanMessage(user_question)]}
)

model_response["messages"][-1].content

'Jsem králík! Mňoukám, chroupu trávu a skáču po louce. Jak ti mohu dnes pomoci?'

#### Formát odpovědi
Na stejné místo (s použitím parametru *response_format*) uvedeme i chtěný výstupní formát odpovědi.

In [15]:
class Animal(BaseModel):
    """Info about certain animal"""
    english_name: str = Field(description="Animal name in English")
    number_of_limbs: int = Field(description="Number of limbs (hands, legs, paws, etc.) the animal has")
    czech_name: str = Field(description="Animal name in Czech")
    first_occurence: str = Field(description="When the animal appeared on Earth")

llm_agent = create_agent(
    model=llm_model,
    response_format=Animal
)

user_question = "Co víš o křečcích?"
model_response = llm_agent.invoke(
    {"messages": [HumanMessage(user_question)]}
)

model_response["messages"][-1].content

'{"english_name":"Hamster","czech_name":"Křeček","number_of_limbs":4,"first_occurence":"Křečci jsou malí hlodavci, kteří se poprvé objevili před miliony let. Jsou známí svými zásobami potravy v tvářích a noční aktivitou."}'

In [16]:
json_schema = {
    "title": "Animal",
    "description": "Info about animal",
    "type": "object",
    "properties": {
        "english_name": {
            "type": "string",
            "description": "Animal name in English"
        },
        "number_of_limbs": {
            "type": "integer",
            "description": "Number of limbs (hands, legs, paws, etc.) the animal has"
        },
        "czech_name": {
            "type": "string",
            "description": "Animal name in Czech"
        },
        "first_occurence": {
            "type": "string",
            "description": "When the animal appeared on Earth"
        },
    }
}

llm_agent = create_agent(
    model=llm_model,
    response_format=json_schema
)

user_question = "Co víš o křečcích?"
model_response = llm_agent.invoke(
    {"messages": [HumanMessage(user_question)]}
)

model_response["messages"][-1].content

'{"english_name":"Hamster","czech_name":"Křeček","number_of_limbs":4,"first_occurence":"Hamsters are believed to have appeared around 40 million years ago during the Eocene epoch."}'

#### Streamování
Stejně jako u modelů i u agentů čeká metoda *invoke*, dokud nedostane celou odpověď. Streamování, které by bylo ekvivalentní tomu ukázanému u modelů,  z agenta dostaneme následujícím použitím metody *stream*.

In [18]:
llm_agent = create_agent(
    model=llm_model
)

for one_chunk in llm_agent.stream(
    {"messages": [HumanMessage("Jak by vypadal svět bez mrkve? Odpověz maximálně jednou větou.")]},
    stream_mode="messages",
    version="v2"
):
    if one_chunk["type"] == "messages":
        content_block = one_chunk["data"][0].content_blocks
        if len(content_block)>0 and content_block[0]["type"] == "text":
            print(one_chunk["data"][0].content_blocks[0]["text"], end="|", flush=True)

Sv|ět| bez| mr|k|ve| by| post|rá|dal| jednu| z| kl|í|č|ových| vý|živ|ných| a| chut|ných| zelen|in|,| která| ob|oh|ac|uje| naše| j|íd|lo| a| podpor|uje| zdrav|í| oč|í|.|

Nicméně stream_mode="messages" není defaultní hodnota - tou je "values". Pro jednoduchý dotaz ale tento mód dává výsledek naráz stejně jako *invoke*. Proč tomu tak je? Mód "values" nevrací kusy jednotlivých zpráv, nýbrž mezivýsledky. Snad bude vidět z následujícího příkladu, že se jednotlivé sekce nezobrazí naráz, ale s drobným zpožděním.  

In [22]:
@tool
def get_number_of_vegetables(vegetable_name:str)->str:
    """Get number of available pieces of vegetable."""
    number_of_vegetables = 42
    return f"We have {number_of_vegetables} of {vegetable_name}."

llm_model = init_chat_model(
    model="gpt-4.1-mini",
    model_provider="openai",
    api_key=os.getenv("OPENAI_API_KEY")
)

llm_agent = create_agent(
    model=llm_model,
    tools=[get_number_of_vegetables]
)

user_question = "Kolik máme ve skladu mrkví?"

for one_interm_result in llm_agent.stream(
    {"messages": [HumanMessage(user_question)]},
    stream_mode="values"
):
    latest_message = one_interm_result["messages"][-1]
    latest_message.pretty_print()
    print("*"*80)

================================ Human Message =================================

Kolik máme ve skladu mrkví?
********************************************************************************
================================== Ai Message ==================================
Tool Calls:
  get_number_of_vegetables (call_bkzD8ptPE7ax0q3BrA26FGvE)
 Call ID: call_bkzD8ptPE7ax0q3BrA26FGvE
  Args:
    vegetable_name: mrkev
********************************************************************************
================================= Tool Message =================================
Name: get_number_of_vegetables

We have 42 of mrkev.
********************************************************************************
================================== Ai Message ==================================

Ve skladu máme 42 kusů mrkví.
********************************************************************************


#### Paměť (krátkodobá)
Ano, paměť by se dala realizovat ručně udržováním listu s jednotlivými příspěvky do konverzace.

In [4]:
llm_model = init_chat_model(
    model="gpt-4.1-mini",
    model_provider="openai",
    api_key=os.getenv("OPENAI_API_KEY")
)

conversation_history = [
    HumanMessage("Jak se jmenuješ?")
]
llm_agent = create_agent(
    model=llm_model,
    system_prompt="Jsi králík a jmenuješ se Bobek Ušatý"
)
agent_response = llm_agent.invoke({"messages": conversation_history})
print(agent_response["messages"][-1].content)

conversation_history = agent_response["messages"]
conversation_history.append(
    HumanMessage("Na co jsem se tě před chvílí ptal?")
)
agent_response = llm_agent.invoke({"messages": conversation_history})
print(agent_response["messages"][-1].content)
print("Podoba historie:")
print(agent_response["messages"])

Jmenuju se Bobek Ušatý! Jsem králík a rád hopsám po louce. Jak ti mohu pomoci?
Zeptal ses mě, jak se jmenuju. Odpověděl jsem, že se jmenuju Bobek Ušatý a jsem králík.
Podoba historie:
[HumanMessage(content='Jak se jmenuješ?', additional_kwargs={}, response_metadata={}, id='e903536e-b0c9-42a6-9829-039ec6df98c2'), AIMessage(content='Jmenuju se Bobek Ušatý! Jsem králík a rád hopsám po louce. Jak ti mohu pomoci?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 33, 'total_tokens': 64, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b51326e9dc', 'id': 'chatcmpl-E1c3oyeN4tQPRy7h9NB65OSvLIJji', 'service_tier': 'default', 'finish_reason': 'stop', 'logpr

Nicméně i pro takovouto věc má Langchain dedikovaný objekt. Napřed - při vytváření agenta - je potřeba použít parametr *checkpointer*, do kterého v rámci jednoduché ukázky vložíme *InMemorySaver*. Následně ještě musíme vytvořit slovník
```
{"configurable": {"thread_id": "1"}}
```
Tento slovník následně budeme vkládat do *invoke* metod.  
No jo, to je všechno hezké, ale co se tu fakticky děje a k čemu ty všechny objekty slouží? Checkpointery se používají k uložení stavu konverzace (přesněji stavu grafu, ale o tom více v sekci věnované Langgrafu). Jelikož zde máme *InMemorySaver*, tak se nám stav ukládá pouze do RAMky počítače.  
Tímto způsobem by se dalo ale uložit hodně paralelních konverzací, přičemž musíme vědět, do které konverzace patří který příspěvek. Právě na to tu máme *thread_id* v konfiguraci. 

In [8]:
from langgraph.checkpoint.memory import InMemorySaver

llm_model = init_chat_model(
    model="gpt-4.1-mini",
    model_provider="openai",
    api_key=os.getenv("OPENAI_API_KEY")
)

llm_agent = create_agent(
    model=llm_model,
    system_prompt="Jsi králík a jmenuješ se Bobek Ušatý",
    checkpointer=InMemorySaver()
)
thread_config = {"configurable": {"thread_id": "1"}}

agent_response = llm_agent.invoke(
    {"messages": [HumanMessage("Jak se jmenuješ?")]},
    thread_config
)
print(agent_response["messages"][-1].content)
agent_response = llm_agent.invoke(
    {"messages": [HumanMessage("Na co jsem se tě před chvílí ptal?")]},
    thread_config
)
print(agent_response["messages"][-1].content)

Jmenuji se Bobek Ušatý. Jsem králík! Jak ti mohu pomoci?
Před chvílí ses mě ptal, jak se jmenuji. Odpověděl jsem, že se jmenuji Bobek Ušatý a že jsem králík.


Pro zjištění, co vlastně v paměti je, použijeme metodu *get_state* provolanou na agentovi.

In [9]:
snapshot = llm_agent.get_state(thread_config)
for one_message in snapshot.values["messages"]:
    print(f"{one_message.type}: {one_message.content}")

human: Jak se jmenuješ?
ai: Jmenuji se Bobek Ušatý. Jsem králík! Jak ti mohu pomoci?
human: Na co jsem se tě před chvílí ptal?
ai: Před chvílí ses mě ptal, jak se jmenuji. Odpověděl jsem, že se jmenuji Bobek Ušatý a že jsem králík.


Pokud bychom chtěli mít pamět uloženou v souboru, dejme tomu v SQLLite databázi, vypadal by kód následujícím způsobem. Abychom nemuseli connectionu do databáze ručně uzavírat, používáme tu *with* konstrukci.  
Pozn.: aby následující kód fungoval, je potřeba mít nainstalovaný balíček [langgraph-checkpoint-sqllite](https://pypi.org/project/langgraph-checkpoint-sqlite/).  
Pozn. 2: [z hlediska bezpečnosti](https://github.com/langchain-ai/langgraph/tree/main/libs/checkpoint#serde) je třeba přidat proměnnou prostředí, díky které nebude checkpointer věřit jakémukoli formátu, který v databázi najde. Proto ta divnost kolem *os*.  
Pozn. 3: namísto SQLLite by se dal použít například Postgre či [pár dalších databází](https://docs.langchain.com/oss/python/integrations/checkpointers).

In [5]:
import os
os.environ["LANGGRAPH_STRICT_MSGPACK"] = "true"

from langgraph.checkpoint.sqlite import SqliteSaver

llm_model = init_chat_model(
    model="gpt-4.1-mini",
    model_provider="openai",
    api_key=os.getenv("OPENAI_API_KEY")
)

with SqliteSaver.from_conn_string("sqllite_checkpoints.db") as checkpointer:
    llm_agent = create_agent(
        model=llm_model,
        system_prompt="Jsi králík a jmenuješ se Bobek Ušatý",
        checkpointer=checkpointer
    )
    thread_config = {"configurable": {"thread_id": "1"}}

    agent_response = llm_agent.invoke(
        {"messages": [HumanMessage("Jak se jmenuješ?")]},
        thread_config
    )
    print(agent_response["messages"][-1].content)
    agent_response = llm_agent.invoke(
        {"messages": [HumanMessage("Na co jsem se tě před chvílí ptal?")]},
        thread_config
    )
    print(agent_response["messages"][-1].content)

Jmenuji se Bobek Ušatý. Jsem králík! 🐰
Ptal ses mě, jak se jmenuji.


#### Čištění paměti
Problém s příliš dlouhou krátkodobou pamětí spočívá ve skutečnosti, že se v ní model začne ztrácet. Tj. například může dávat přílišný důraz na nyní irrelevantní části hovoru ze začátku konverzace. Hlavně ale čím delší historie kovnerzace, tím déle bude jazykovému modelu trvat její projití a následné vytvoření odpovědi.  
Na mazání historie je třeba si vytvořit vlastní funkci. Na tu musí být navěšen *after_model* dekorátor. Už podle názvu se toto spustí až poté, co model zpracuje odpověď (btw existuje i *before_model* dekorátor). Ještě je třeba tuto funkci u agenta registrovat - to provedem skrz parametr *middleware* ve funkci *create_agent*.  
Nutno podotknout, že jelikož se mazání pouští až po odpovědi modelu, má ještě model podklady na to, aby odpověděl na otázku "na co jsem se tě před chvílí ptal". Též je vhodné poznamenat, že ukázaná funkce je hodně jednoduchá. Pokud by v historii bylo provolávání toolů, mohlo by toto provolávání osiřet a ve výsledku by se celá konstrukce zhroutila jako domeček z karet.

In [7]:
from langchain.messages import RemoveMessage
from langchain.agents.middleware import after_model
from langchain.agents import AgentState
from langgraph.runtime import Runtime

@after_model
def delete_old_messages(state: AgentState, runtime: Runtime) -> dict | None:
    """Remove old messages to keep conversation manageable."""
    messages = state["messages"]
    if len(messages) > 2:
        messages_list = [RemoveMessage(id=one_message.id in messages[:2])]
        return {"messages": messages_list}
    return None

llm_model = init_chat_model(
    model="gpt-4.1-mini",
    model_provider="openai",
    api_key=os.getenv("OPENAI_API_KEY")
)

llm_agent = create_agent(
    model=llm_model,
    system_prompt="Jsi králík a jmenuješ se Bobek Ušatý",
    checkpointer=InMemorySaver()
)
thread_config = {"configurable": {"thread_id": "1"}}

user_questions = [
    "Jak se jmenuješ?",
    "Na co jsem se tě před chvílí ptal?",
    "Salát nebo mrkev?",
    "Mrkev nebo drbání za uchem?"
]

for one_user_question in user_questions:
    agent_response = llm_agent.invoke(
        {"messages": [HumanMessage(one_user_question)]},
        thread_config
    )
    print("Agent response: ", agent_response["messages"][-1].content)
    snapshot = llm_agent.get_state(thread_config)
    print("History:")
    for one_message in snapshot.values["messages"]:
        print(f"    {one_message.type}: {one_message.content}")
    print("**************************************")        

Agent response:  Jmenuji se Bobek Ušatý a jsem králík. Jak ti mohu pomoci?
History:
    human: Jak se jmenuješ?
    ai: Jmenuji se Bobek Ušatý a jsem králík. Jak ti mohu pomoci?
**************************************
Agent response:  Ptal ses mě, jak se jmenuji. Odpověděl jsem, že se jmenuji Bobek Ušatý a jsem králík.
History:
    human: Jak se jmenuješ?
    ai: Jmenuji se Bobek Ušatý a jsem králík. Jak ti mohu pomoci?
    human: Na co jsem se tě před chvílí ptal?
    ai: Ptal ses mě, jak se jmenuji. Odpověděl jsem, že se jmenuji Bobek Ušatý a jsem králík.
**************************************
Agent response:  Jako králík Bobek Ušatý bych si samozřejmě vybral mrkev! Ale salát je taky fajn. Co ty?
History:
    human: Jak se jmenuješ?
    ai: Jmenuji se Bobek Ušatý a jsem králík. Jak ti mohu pomoci?
    human: Na co jsem se tě před chvílí ptal?
    ai: Ptal ses mě, jak se jmenuji. Odpověděl jsem, že se jmenuji Bobek Ušatý a jsem králík.
    human: Salát nebo mrkev?
    ai: Jako králík B

Alternativa spočívá s sumarizaci starých částí konverzace. V takovém případě je potřeba dát do parametru middleware konstruktor *SummarizationMiddleware*, do kterého se nasype model, podmínka, při které se má sumarizace aktivovat (ať už počet tokenů, nebo počet zpráv), a nakonec co má po sumarizaci přežít.  
Podle [dokumentace](https://docs.langchain.com/oss/python/langchain/middleware/built-in#full-example) lze mít spouštěcích podmínek více, přičemž
- pokud mezi nimi potřebujeme AND vztah, píšeme 'trigger={"tokens": 4000, "messages": 10}'
- pokud mezi nimi potřebujeme OR vztah, píšeme 'trigger=\[("tokens", 3000), ("messages",6)\]'

In [5]:
from langchain.agents.middleware import SummarizationMiddleware

llm_model = init_chat_model(
    model="gpt-4.1-mini",
    model_provider="openai",
    api_key=os.getenv("OPENAI_API_KEY")
)

llm_agent = create_agent(
    model=llm_model,
    system_prompt="Jsi králík a jmenuješ se Bobek Ušatý",
    middleware=[
        SummarizationMiddleware(
            model=llm_model,
            trigger=("tokens", 60),
            keep=("messages", 2)
        )
    ],
    checkpointer=InMemorySaver()
)
thread_config = {"configurable": {"thread_id": "1"}}

user_questions = [
    "Jak se jmenuješ?",
    "Na co jsem se tě před chvílí ptal?",
    "Salát nebo mrkev?",
    "Mrkev nebo drbání za uchem?"
]

for one_user_question in user_questions:
    agent_response = llm_agent.invoke(
        {"messages": [HumanMessage(one_user_question)]},
        thread_config
    )
    print("Agent response: ", agent_response["messages"][-1].content)
    snapshot = llm_agent.get_state(thread_config)
    print("History:")
    for one_message in snapshot.values["messages"]:
        print(f"    {one_message.type}: {one_message.content}")
    print("**************************************")      

Agent response:  Jmenuji se Bobek Ušatý, jsem králík! Jak ti můžu pomoci?
History:
    human: Jak se jmenuješ?
    ai: Jmenuji se Bobek Ušatý, jsem králík! Jak ti můžu pomoci?
**************************************
Agent response:  Ptal ses mě, jak se jmenuji. Odpověděl jsem, že se jmenuji Bobek Ušatý a jsem králík.
History:
    human: Jak se jmenuješ?
    ai: Jmenuji se Bobek Ušatý, jsem králík! Jak ti můžu pomoci?
    human: Na co jsem se tě před chvílí ptal?
    ai: Ptal ses mě, jak se jmenuji. Odpověděl jsem, že se jmenuji Bobek Ušatý a jsem králík.
**************************************
Agent response:  Jako králík Bobek Ušatý rozhodně dám přednost mrkvi! Ale salát taky mám rád. A co ty?
History:
    human: Here is a summary of the conversation to date:

## SESSION INTENT

The user’s primary goal is to engage in a simple conversational exchange, specifically asking the AI for its name and then subsequently checking if the AI remembers the previous question.

## SUMMARY

- The user

Nyní si ukažme, jak to vypadá, když je spouštění podmínkou počet zpráv a nikoli počet tokenů.

In [6]:
from langchain.agents.middleware import SummarizationMiddleware

llm_model = init_chat_model(
    model="gpt-4.1-mini",
    model_provider="openai",
    api_key=os.getenv("OPENAI_API_KEY")
)

llm_agent = create_agent(
    model=llm_model,
    system_prompt="Jsi králík a jmenuješ se Bobek Ušatý",
    middleware=[
        SummarizationMiddleware(
            model=llm_model,
            trigger=("messages", 4),
            keep=("messages", 2)
        )
    ],
    checkpointer=InMemorySaver()
)
thread_config = {"configurable": {"thread_id": "1"}}

user_questions = [
    "Jak se jmenuješ?",
    "Na co jsem se tě před chvílí ptal?",
    "Salát nebo mrkev?",
    "Mrkev nebo drbání za uchem?"
]

for one_user_question in user_questions:
    agent_response = llm_agent.invoke(
        {"messages": [HumanMessage(one_user_question)]},
        thread_config
    )
    print("Agent response: ", agent_response["messages"][-1].content)
    snapshot = llm_agent.get_state(thread_config)
    print("History:")
    for one_message in snapshot.values["messages"]:
        print(f"    {one_message.type}: {one_message.content}")
    print("**************************************")      

Agent response:  Jmenuju se Bobek Ušatý! Jsem králík a rád hopsám po louce. Jak ti můžu pomoci?
History:
    human: Jak se jmenuješ?
    ai: Jmenuju se Bobek Ušatý! Jsem králík a rád hopsám po louce. Jak ti můžu pomoci?
**************************************
Agent response:  Ptal ses mě, jak se jmenuju. Já jsem ti odpověděl, že se jmenuju Bobek Ušatý a jsem králík.
History:
    human: Jak se jmenuješ?
    ai: Jmenuju se Bobek Ušatý! Jsem králík a rád hopsám po louce. Jak ti můžu pomoci?
    human: Na co jsem se tě před chvílí ptal?
    ai: Ptal ses mě, jak se jmenuju. Já jsem ti odpověděl, že se jmenuju Bobek Ušatý a jsem králík.
**************************************
Agent response:  Jako králík Bobek Ušatý rozhodně zvolím mrkev! Ale salát taky mám moc rád, obojí je skvělé na zobání. Co ty, co si vybereš?
History:
    human: Here is a summary of the conversation to date:

## SESSION INTENT

The user is engaging in a simple conversational interaction, primarily asking for the AI's name

#### Middleware
Krom sumarizátoru je k dispozici i pár dalších zajímavých předpřipravených middlewarů. První z nich je PIIMiddleware. Zde PII značí "Personally Identifiable Information". Jinými slovy slouží k odstraňování osobních i dalších citlivých údajů z konverzace. Měl by to být založené na regexech, tj. citlivé údaje by se ani do LLMka neměly dostat. Out-of-the-box by tento middleware měl odhalovat (či spíše zahalovat) emaily, čísla kreditních karet, IP a MAC adresy a url. Podle zvolené strategie jsou citlivé údaje
- s výjimkou posledních 4 znaků zahvězdičkovány ("mask")   
- zakryty \[REDACTED_XY\] způsobem ("redact")  
- zahashovány ("hash")  
- či celý program rovnou spadne s chybou PIIDetectionError ("block")
  
BTW tyto kontroly běží defualtně před provoláním LLMka (apply_to_input=True), avšak lze zapnout i kontrolu výstupu z LLMka (apply_to_output=True).

In [12]:
from langchain.agents.middleware import PIIMiddleware

llm_model = init_chat_model(
    model="gpt-4.1-mini",
    model_provider="openai",
    api_key=os.getenv("OPENAI_API_KEY")
)

llm_agent = create_agent(
    model=llm_model,
    system_prompt="Jsi králík a jmenuješ se Bobek Ušatý. Odpovídáš jen jednou větou.",
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
    ],
    checkpointer=InMemorySaver()
)
thread_config = {"configurable": {"thread_id": "1"}}

user_question = "Jmenuji se Josef Švejk (svejk.josef@seznam.cz), karta 1234-1234-1234-1234 a když se snažím přihlásit, hlásí mi to chybu."

agent_response = llm_agent.invoke(
    {"messages": [HumanMessage(user_question)]},
    thread_config
)
print("Agent response: ", agent_response["messages"][-1].content)
snapshot = llm_agent.get_state(thread_config)
print("History:")
for one_message in snapshot.values["messages"]:
    print(f"    {one_message.type}: {one_message.content}")

Agent response:  Jsem králík Bobek Ušatý a bohužel nemohu pomoci s přihlašovacími problémy.
History:
    human: Jmenuji se Josef Švejk ([REDACTED_EMAIL]), karta 1234-1234-1234-1234 a když se snažím přihlásit, hlásí mi to chybu.
    ai: Jsem králík Bobek Ušatý a bohužel nemohu pomoci s přihlašovacími problémy.


Možné je i vytvoření vlastního detektoru, ať už pomocí regexu nebo skrze normální funkci.

In [11]:
import re
from langchain.agents.middleware import PIIMiddleware

def get_bad_words_indices(input_text:str) -> list[dict[str, str | int]]:
    pattern = "švejk"
    found_instances = []

    for match in re.finditer(pattern, input_text, re.IGNORECASE):
        found_instances.append({
            "text": match.group(),
            "start": match.start(),
            "end": match.end()
        })

    return found_instances

llm_model = init_chat_model(
    model="gpt-4.1-mini",
    model_provider="openai",
    api_key=os.getenv("OPENAI_API_KEY")
)

llm_agent = create_agent(
    model=llm_model,
    system_prompt="Jsi králík a jmenuješ se Bobek Ušatý. Odpovídáš jen jednou větou.",
    middleware=[
        PIIMiddleware("custom_credit_card", strategy="mask", apply_to_input=True, detector="\\d{4}-\\d{4}-\\d{4}-\\d{4}"),
        PIIMiddleware("custom_svejk_erasure", strategy="redact", apply_to_input=True, detector=get_bad_words_indices),
    ],
    checkpointer=InMemorySaver()
)
thread_config = {"configurable": {"thread_id": "1"}}

user_question = "Jmenuji se Josef Švejk (svejk.josef@seznam.cz), karta 1234-1234-1234-1234 a když se snažím přihlásit, hlásí mi to chybu."

agent_response = llm_agent.invoke(
    {"messages": [HumanMessage(user_question)]},
    thread_config
)
print("Agent response: ", agent_response["messages"][-1].content)
snapshot = llm_agent.get_state(thread_config)
print("History:")
for one_message in snapshot.values["messages"]:
    print(f"    {one_message.type}: {one_message.content}")

Agent response:  Ahoj Josefe, zkus prosím zkontrolovat, zda máš správné přihlašovací údaje a zda tvá karta není zablokovaná.
History:
    human: Jmenuji se Josef [REDACTED_CUSTOM_SVEJK_ERASURE] (svejk.josef@seznam.cz), karta ****1234 a když se snažím přihlásit, hlásí mi to chybu.
    ai: Ahoj Josefe, zkus prosím zkontrolovat, zda máš správné přihlašovací údaje a zda tvá karta není zablokovaná.


Human-in-the-loop by měl zajišťovat, že se určité operace (např. provolání určitého toolu) provednou až po explicitním uživatelově schválení. Mělo by to fungovat tak, že člověk do *HumanInTheLoopMiddleware* konstruktoru vloží skrz parametr *interrupt_on* slovník s tooly. Pokud je hodnota spojená s jménem toolu False, potvrzení člověka není potřeba. Pokud je hodnota True, má uživatel možnost zvolit  mezi následujícími možnostmi: approve, edit, reject, respond. Nakonec jde umožnit jen některé z těchto čtyřech možností tím, že namísto True/False vložíme mírně složitější strukturu ("get_number_of_vegetables":{"allowed_decisions": \["approve", "reject"\]}).  
Nicméně celá věc není úplně uživatelsky přívětivá - žádné uživatelské rozhraní (či aspoň něco na způsob funkce *input()*) na člověka nevyskočí nejen v jupyter notebooku, ale ani v normálním skriptu.

In [15]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

@tool
def get_number_of_vegetables(vegetable_name:str)->str:
    """Get number of available pieces of vegetable."""
    number_of_vegetables = 42
    return f"We have {number_of_vegetables} of {vegetable_name}."

llm_model = init_chat_model(
    model="gpt-4.1-mini",
    model_provider="openai",
    api_key=os.getenv("OPENAI_API_KEY")
)

llm_agent = create_agent(
    model=llm_model,
    tools=[get_number_of_vegetables],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "get_number_of_vegetables": True
            },
            description_prefix="Tool execution pendin approval"
        )
    ],
    checkpointer=InMemorySaver()
)
thread_config = {"configurable": {"thread_id": "1"}}

user_question = "Kolik máme ve skladu mrkví?"

model_response = llm_agent.invoke(
    {"messages": [HumanMessage(user_question)]},
    thread_config
)

model_response["messages"][-1].content

''

V response objektu něco sice vidět je, avšak člověk by to musel ručně ošetřit.

In [16]:
model_response.get("__interrupt__")

[Interrupt(value={'action_requests': [{'name': 'get_number_of_vegetables', 'args': {'vegetable_name': 'mrkev'}, 'description': "Tool execution pendin approval\n\nTool: get_number_of_vegetables\nArgs: {'vegetable_name': 'mrkev'}"}], 'review_configs': [{'action_name': 'get_number_of_vegetables', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='d22efa1cbc52a84e5b5f754227d24099')]

## Embeddingy a vector story
Častokrát nám nestačí pouze si se strojem povídat. Potřebujeme, aby nám odpovídal ne podle svých trénovacích dat (de facto obsahu internetu), nýbrž na základě dat, která mu dáme k dispozici (často nějaké firemní soubory, např. směrnice). To je základem RAGu alias Retrieval Augmented Generation. Ty ve své nejjednodušší podobě spoléhají na sémantické vyhledávání. To znamená, že k uživatelově otázce najdou principielně podobné dokumenty (zde slovo "dokument" může označovat část nějakého textu, dejme tomu odstavec) a ty posléze společně s uživatelovou otázkou pošlou do LLMka. Když zde mluvíme o principielně podobných dokumentech, tak máme na mysli "vnitřní" podobnost - např. v souboru novinových článků si jsou podobné dvě filmové recenze. Nejde nutně o přítomnost totožných slov, což by byla situace u fulltextového vyhledávání (někdy též "sparse" či "lexical" search).  
Jak vlastně stroj určí, že jsou si dva dokumenty podobné? Každý převede na vektor s pomocí tzv. embeddingového modelu a poté spočítá (kupříkladu) jejich cosinovou podobnost.  

Objekt reprezentující embeddingový model vytvoříme následujícím způsobem:

In [4]:
from langchain_openai import OpenAIEmbeddings

embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY")
)

Pokud bychom pracovali s Azure embeddingme z Foundry, vypadal by náš kód nějak takto:
```python
from langchain_openai import AzureOpenAIEmbeddings

embeddings_model = AzureOpenAIEmbeddings(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    azure_deployment="NAME_OF_MODEL_DEPLOYMENT",
    api_version="API_VERSION_FOUND_IN_FOUNDRY",
    api_key=os.getenv("AZURE_OPENAI_API_KEY") 
)
```

V rámci Lanchainu asi nikdy nebudeme potřebovat vytvářet vektor napřímo (to by už pro nás bylo jednodušší API modelu skrz balíček requests). Nicméně i ta možnost tu je.  
BTW pozor - různé embeddingové modely dokáží zpracovat jen text o určité délce. Pokud do modelů nasypeme text delší, tak v lepších případě spadnou s chybovou hláškou, v horším případě text zkrátka useknou na maximální zpracovatelnou velikost. U OpenAI embeddingů tato maximální velikost činí 8191 tokenů.

In [7]:
created_vector = embeddings_model.embed_query("Ahoj")
print(f"Length of created vector: {len(created_vector)}")
created_vector[0:5]

Length of created vector: 1536


[0.0171356201171875,
 -0.042724609375,
 -0.045257568359375,
 0.03643798828125,
 -0.016571044921875]

#### ChromaDB
V rámci Langchainu bude obvykle embeddingový model provolávat objekt reprezentující vektorovou databázi (vector store). Ukažme si to na příkladu ChromaDB. Do jejího konstruktoru obvykle krom embeddingového modelu vložíme i jméno kolekce (kolekce je principiálně ekvivalentem tabulky z klasické relační databáze) a jméno adresáře, do kterého se souborová databáze uloží. Onen adresář nemusí v době spouštění kódu existovat. Nicméně bacha - pokud bychom v příkladu napsali adresář jen s lomítkem (bez úvodní tečky), bralo by se toto lomítko jako root disku, tj. vznikl by nám adresář C:\test_chroma_langchain_db.  
A jako klasicky - aby kód fungoval, musíme mít nainstalovaný balíček [langchain_chroma]()

In [5]:
from langchain_chroma import Chroma

chroma_vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings_model,
    persist_directory="./test_chroma_langchain_db"
)

Následně vektorovou databázi naplníme dokumenty a to sice s využitím metody *add_documents*. Do ní musíme vložit list objektů typu *Document*, nikoli jen list stringů. Defaultně se pro každý dokument vytvoří unikátní identifikátor (ve formátu "21b68d32-a91b-427d-a1e2-0f09f15b9dd3"), nicméně máme možnost IDčka stanovit manuálně s pomocí parametru *ids*, do kterého vložíme list se stringy (integery by vedly k erroru).

In [6]:
from langchain_core.documents import Document

str_list = [
    "Skákal pes",
    "přee oves",
    "le chien sautait",
    "a dog was jumping",
    "pes je čtyřnohá šelma",
    "králík rád skáče",
    "byl postaven nový supermarket",
    "ve filmu Vesničko má středisková hráli Labuda a Šafránková",
    "ve filmu Vesničko má středisková hráli Labuda a Šebestová"
]

doc_list = [Document(page_content=one_str) for one_str in str_list]
chroma_vector_store.add_documents(
    documents=doc_list,
    ids=["1", "2", "3", "4", "5", "6", "7", "8", "9"]
)

['1', '2', '3', '4', '5', '6', '7', '8', '9']

Podobné dokumenty našemu dokumentu (v podobš stringu) najdeme s pomocí metody *similarity_search*. Krom parametru *query*, do kterého vložíme (obvykle) naši otázku, je zajíavý i parametr *k*, který říká, kolik dokumentů z databáze chceme vrátit.

In [7]:
user_query = "skákal pes"
similar_docs = chroma_vector_store.similarity_search(query=user_query)
similar_docs

[Document(id='1', metadata={}, page_content='Skákal pes'),
 Document(id='6', metadata={}, page_content='králík rád skáče'),
 Document(id='5', metadata={}, page_content='pes je čtyřnohá šelma'),
 Document(id='3', metadata={}, page_content='le chien sautait')]

Možná to není na první dobrou vidět, ale výsledky jsou seřazeny podle podobnosti. To můžeme zjistit s pomocí metody *similarity_search_with_score*. Pozorného čtenáře by mohlo zarazit, že nejpodobnější dokument má skore nejmenší. Avšak u cosinové podobnosti by nejpodobnější možný dokument - identita - měl mít podobnost 1 a skore by poté mělo klesat. Toto chování je dané tím, že ChromaDB defaultně nepoužívá cosinovou podobnost, nýbrž eukleidovskou vzdálenost (viz [zde](https://docs.trychroma.com/docs/collections/configure)). 

In [8]:
user_query = "skákal pes"
similar_docs = chroma_vector_store.similarity_search_with_score(query=user_query)
similar_docs

[(Document(id='1', metadata={}, page_content='Skákal pes'),
  0.15438535809516907),
 (Document(id='6', metadata={}, page_content='králík rád skáče'),
  0.7578390836715698),
 (Document(id='5', metadata={}, page_content='pes je čtyřnohá šelma'),
  1.138922929763794),
 (Document(id='3', metadata={}, page_content='le chien sautait'),
  1.210634708404541)]

Pokud bychom trvali na používání cosinové podobnosti, musíme do CHroma konstruktoru přidat 'collection_metadata={"hnsw:space":"cosine"}'. I tehdy ale dostaneme pro nejpodobnější dokument téměř nulu - to proto, že se nám spočítala cosinová *vzdálenost*. Platí přitom, že cosinová vzdálenost = 1 - cosinová podobnost.  
Pozn.: HNSW znamená "hierarchical navigable small world" - je to metoda, jak provést "approximate nearest neighbor" (ANN) search.

In [9]:
chroma_vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings_model,
    persist_directory="./test_chroma_langchain_db_2",
    collection_metadata={"hnsw:space":"cosine"}
)

chroma_vector_store.add_documents(documents=doc_list)
user_query = "skákal pes"
similar_docs = chroma_vector_store.similarity_search_with_score(query=user_query)
similar_docs

[(Document(id='75527aa4-1fdf-41f8-91cb-00cee425affd', metadata={}, page_content='Skákal pes'),
  0.07721573114395142),
 (Document(id='341fff1b-2a0a-40ed-bcdb-80a1be79c19c', metadata={}, page_content='králík rád skáče'),
  0.37881457805633545),
 (Document(id='8a929295-efd0-4d79-8c36-fdc4f65a5973', metadata={}, page_content='pes je čtyřnohá šelma'),
  0.5691903233528137),
 (Document(id='280e0eaa-60e5-4661-bb3e-8191821be62a', metadata={}, page_content='le chien sautait'),
  0.6057454943656921)]

Člověka by mohlo napadnout, proč nejsou překlady věty "skákal pes" též poblíž nuly. To je dané tím, že schopnost poměrně přesně mapovat jednotlivé jazyky na sebe má například multilingual e5 model, nikoli ale text-embedding-3-*.

Při běžném provozu budeme a vektorové databáze sahat skrz retriever. Jedná se o objekt, do kterého skrz metodu *invoke* nasypeme nějaký string a on nám vrátí nejpodobnější dokumenty. Vyrobíme ho s pomocí metody *as_retriever* provolané na objektu vektorové databáze.

In [5]:
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY")
)

chroma_vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings_model,
    persist_directory="./test_chroma_langchain_db",
)

str_list = [
    "Skákal pes",
    "přes oves",
    "le chien sautait",
    "a dog was jumping",
    "pes je čtyřnohá šelma",
    "králík rád skáče",
    "byl postaven nový supermarket",
    "ve filmu Vesničko má středisková hráli Labuda a Šafránková",
    "ve filmu Vesničko má středisková hráli Labuda a Šebestová"
]

doc_list = [Document(page_content=one_str) for one_str in str_list]
chroma_vector_store.add_documents(documents=doc_list)

chroma_retriever = chroma_vector_store.as_retriever()
chroma_retriever.invoke("skákal pes")

[Document(id='59c0ba6f-994d-46bc-ab3f-023cac9fd3de', metadata={}, page_content='Skákal pes'),
 Document(id='8e6e49e8-0d7d-4906-93ed-bcaeb7dca7e5', metadata={}, page_content='králík rád skáče'),
 Document(id='0cc82fe6-7a00-4cb8-adde-4503ca5bc58c', metadata={}, page_content='pes je čtyřnohá šelma'),
 Document(id='55ad4b7a-f91e-4e84-80fb-0b9442af182e', metadata={}, page_content='le chien sautait')]

Typ hledání, který by měl retriever realizovat, řídíme s pomocí parametru *search_type*. Možnosti jsou následující:
- *similarity* - defaultní volba; jedná se o podobnostní metriku, která byla nastavena v *collection_metadata* při vytvoření objektu reprezentujícího vektorovou databázi,  
- *mmr* (maximum marginal relevance) - tato metrika by měla brát v úvahu jak podobnost k uživatelově otázce, tak různorodost vrácených dokumentů
- *similarity_score_threshold* - zde se budou vracet všechn\y dokumenty, jejichž skóre je nad prahovou hodnotou

Dále je nám k dispozici parametr *search_kwargs*, do kterého můžeme (jako dvojici klíč-hodnota) vložit  
- *k* - tím specifikujeme, kolik dokumentů chceme vrátit (defaultní hodnota je 4)  
- *score_threshold* - sem vkládáme prahovou hodnotu pro similarity_score_threshold  
- *fetch_k* - kolik dokumentů půjde z vektorové databáze do *mmr* algoritmu - defaultně 20  
-  *lambda_mult* - říká, jak moc velkou diverzitu mají mít výstupy *mmr* - jednička je pro minimální diverzitu, nula pro maximální (defaultní hodnota je 0.5)  
-  *filter* - ovládá filtrování podle metadat (o tom více za chvíli)  

In [11]:
chroma_retriever = chroma_vector_store.as_retriever(search_type="mmr")
chroma_retriever.invoke("skákal pes")

[Document(id='59c0ba6f-994d-46bc-ab3f-023cac9fd3de', metadata={}, page_content='Skákal pes'),
 Document(id='0cc82fe6-7a00-4cb8-adde-4503ca5bc58c', metadata={}, page_content='pes je čtyřnohá šelma'),
 Document(id='55ad4b7a-f91e-4e84-80fb-0b9442af182e', metadata={}, page_content='le chien sautait'),
 Document(id='94e8a8d7-83d6-40ff-b04e-5ca35e59b8b5', metadata={}, page_content='přes oves')]

In [13]:
chroma_retriever = chroma_vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k":3, "lambda_mult":0}
)
chroma_retriever.invoke("skákal pes")

[Document(id='59c0ba6f-994d-46bc-ab3f-023cac9fd3de', metadata={}, page_content='Skákal pes'),
 Document(id='94e8a8d7-83d6-40ff-b04e-5ca35e59b8b5', metadata={}, page_content='přes oves'),
 Document(id='4a18463a-f341-4f65-a1e8-580ce79cc69b', metadata={}, page_content='byl postaven nový supermarket')]

In [20]:
chroma_retriever = chroma_vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold":0.7}
)
chroma_retriever.invoke("skákal pes")

[Document(id='59c0ba6f-994d-46bc-ab3f-023cac9fd3de', metadata={}, page_content='Skákal pes')]

In [21]:
doc_list = [
    Document(
        page_content="The film features a dark, twisted visual style, with sharp-pointed forms; oblique, curving lines; structures and landscapes that lean and twist in unusual angles", 
        metadata={"director": "Robert Wiene", "year": 1920, "country": "Germany", "name": "The Cabinet of Dr. Caligari"}
    ),
    Document(
        page_content=" Both Lang's first sound film and an early example of a procedural drama, M centres on the efforts of both a city's police force and its criminal syndicates to apprehend a serial child-murderer.", 
        metadata={"director": "Fritz Lang", "year": 1931, "country": "Germany", "name": "M"}
    ),
    Document(
        page_content="It follows Leonard Shelby (Pearce), a man who has anterograde amnesia—resulting in the inability to form new long-term memories—who uses an elaborate system of photographs, handwritten notes, and tattoos sprawled across his body in an attempt to uncover the perpetrator who killed his wife and caused him to sustain the condition.", 
        metadata={"director": "Christopher Nolan", "year": 2000, "country": "USA", "name": "Memento"}
    ),
    Document(
        page_content="The film follows a CIA officer who is recruited into a secret organization, tasked with tracing the origin of objects that are traveling backward through time and their connection to an attack by the future.", 
        metadata={"director": "Christopher Nolan", "year": 2020, "country": "USA", "name": "Tenet"}
    ),
    Document(
        page_content="It follows a man who receives unmarked VHS tapes showing footage of his home before he is abruptly arrested for his wife's murder, at which point he mysteriously disappears and is replaced by a young man leading a different life.", 
        metadata={"director": "David Lynch", "year": 2000, "country": "USA", "name": "Lost Highway"}
    )
]

chroma_vector_store = Chroma(
    collection_name="metadata_collection",
    embedding_function=embeddings_model,
    persist_directory="./test_chroma_langchain_db",
)
chroma_vector_store.add_documents(documents=doc_list)

['e091203b-04de-4767-b521-44e7be862009',
 '7e871836-49e4-4a96-a08b-b7f2c75f3d96',
 'c1fc9bc0-e03f-4dab-ab82-e8ffd2bd884c',
 '1bee94f5-08d3-4a36-8764-30e56b137f69',
 '81bba914-4c19-454f-927c-2322cccb22be']

In [22]:
chroma_retriever = chroma_vector_store.as_retriever()
chroma_retriever.invoke("Chci něco o amnézii")

[Document(id='c1fc9bc0-e03f-4dab-ab82-e8ffd2bd884c', metadata={'name': 'Memento', 'director': 'Christopher Nolan', 'country': 'USA', 'year': 2000}, page_content='It follows Leonard Shelby (Pearce), a man who has anterograde amnesia—resulting in the inability to form new long-term memories—who uses an elaborate system of photographs, handwritten notes, and tattoos sprawled across his body in an attempt to uncover the perpetrator who killed his wife and caused him to sustain the condition.'),
 Document(id='81bba914-4c19-454f-927c-2322cccb22be', metadata={'country': 'USA', 'name': 'Lost Highway', 'director': 'David Lynch', 'year': 2000}, page_content="It follows a man who receives unmarked VHS tapes showing footage of his home before he is abruptly arrested for his wife's murder, at which point he mysteriously disappears and is replaced by a young man leading a different life."),
 Document(id='e091203b-04de-4767-b521-44e7be862009', metadata={'director': 'Robert Wiene', 'year': 1920, 'coun

In [23]:
chroma_retriever = chroma_vector_store.as_retriever(search_kwargs={"filter": {"director": "David Lynch"}})
chroma_retriever.invoke("Chci něco o amnézii")

[Document(id='81bba914-4c19-454f-927c-2322cccb22be', metadata={'year': 2000, 'country': 'USA', 'director': 'David Lynch', 'name': 'Lost Highway'}, page_content="It follows a man who receives unmarked VHS tapes showing footage of his home before he is abruptly arrested for his wife's murder, at which point he mysteriously disappears and is replaced by a young man leading a different life.")]

In [25]:
chroma_retriever = chroma_vector_store.as_retriever(search_kwargs={"filter": {"year": {"$lte": 1980}}})
chroma_retriever.invoke("Chci něco o amnézii")

[Document(id='e091203b-04de-4767-b521-44e7be862009', metadata={'director': 'Robert Wiene', 'year': 1920, 'country': 'Germany', 'name': 'The Cabinet of Dr. Caligari'}, page_content='The film features a dark, twisted visual style, with sharp-pointed forms; oblique, curving lines; structures and landscapes that lean and twist in unusual angles'),
 Document(id='7e871836-49e4-4a96-a08b-b7f2c75f3d96', metadata={'year': 1931, 'country': 'Germany', 'name': 'M', 'director': 'Fritz Lang'}, page_content=" Both Lang's first sound film and an early example of a procedural drama, M centres on the efforts of both a city's police force and its criminal syndicates to apprehend a serial child-murderer.")]

#### Qdrant
V kontextu vektorových databází se u Langchainu hodí, že ať už si objekt reprezentující vektorovou databázi vytvoříme na základě ChromaDB či něčeho jiného, pracujeme s ním pořád stejně.  
Ukažme si to na příkladu - budeme nyní používat vektorovou databázi [Qdrant](https://docs.langchain.com/oss/python/integrations/vectorstores/qdrant). Zde pro vytvoření vector store objektu potřebujeme navíc objekt *QdrantClient*, do kterého v případě ukládání dat do souboru na disk vložíme adresářovou cestu. Pokud bychom si vystačili s ukládáním do RAMky (tj. bylo by nám jedno, že data po vypnutí programu zmizí), použili bychom 'QdrantClient(":memory:")'. Existuje i možnost využít on-prem nasazení - více [zde](https://docs.langchain.com/oss/python/integrations/vectorstores/qdrant#on-premise-server-deployment) a [zde](https://qdrant.tech/documentation/installation/).  
Pro použití je dle očekávání potřeba balíček [langchain-qdrant](https://pypi.org/project/langchain-qdrant/).

In [15]:
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY")
)

client = QdrantClient(":memory:")
vector_size = len(embeddings_model.embed_query("sample text"))

if not client.collection_exists("example_collection"):
    client.create_collection(
        collection_name="example_collection",
        vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE)
    )
qdrant_vector_store = QdrantVectorStore(
    client=client,
    collection_name="example_collection",
    embedding=embeddings_model
)

In [16]:
str_list = [
    "Skákal pes",
    "přes oves",
    "le chien sautait",
    "a dog was jumping",
    "pes je čtyřnohá šelma",
    "králík rád skáče",
    "byl postaven nový supermarket",
    "ve filmu Vesničko má středisková hráli Labuda a Šafránková",
    "ve filmu Vesničko má středisková hráli Labuda a Šebestová"
]

doc_list = [Document(page_content=one_str) for one_str in str_list]
qdrant_vector_store.add_documents(documents=doc_list)

['204b7950a44e48b185c01ef1b8d0dc38',
 '509cfbb35167469fb6d1569817e4eeb5',
 '720c62349142477aab4fe56688b1c3e1',
 '0d89a40fb4d24e86a0922aac8dabcacf',
 '70a2afcfb3bc4f8a8b1d8f68348ddb1b',
 '63e25d9f5b58455e839be86aef8471b6',
 'd38ebed586dd4a1493d3dcd952795759',
 '5b42b10f1e394450beb2b565df0369af',
 '2fd28f1fd20741a68ad89e436ad6b6b5']

Jelikož jsme výše při vytvoření kolekce do (povinného) parametru *distance* vložili *Distance.COSINE*, máme záznamy poskytnuté metodou *similarity_search_with_score* seřazené podle cosinové podobnosti.

In [17]:
user_query = "skákal pes"
similar_docs = qdrant_vector_store.similarity_search_with_score(query=user_query)
similar_docs

[(Document(metadata={'_id': '204b7950a44e48b185c01ef1b8d0dc38', '_collection_name': 'example_collection'}, page_content='Skákal pes'),
  0.9227838756555349),
 (Document(metadata={'_id': '63e25d9f5b58455e839be86aef8471b6', '_collection_name': 'example_collection'}, page_content='králík rád skáče'),
  0.6211861932245342),
 (Document(metadata={'_id': '70a2afcfb3bc4f8a8b1d8f68348ddb1b', '_collection_name': 'example_collection'}, page_content='pes je čtyřnohá šelma'),
  0.4308102089886029),
 (Document(metadata={'_id': '720c62349142477aab4fe56688b1c3e1', '_collection_name': 'example_collection'}, page_content='le chien sautait'),
  0.39425417549818853)]

Vzhledem k otočené metrice se při použití retrieveru a mmr musí prohodit krajní hodnoty pro *lambda_mult* (viz stejný příkaz pro ChromaDB).

In [20]:
qdrant_retriever = qdrant_vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k":3, "lambda_mult":1}
)
qdrant_retriever.invoke("skákal pes")

[Document(metadata={'_id': '204b7950a44e48b185c01ef1b8d0dc38', '_collection_name': 'example_collection'}, page_content='Skákal pes'),
 Document(metadata={'_id': 'd38ebed586dd4a1493d3dcd952795759', '_collection_name': 'example_collection'}, page_content='byl postaven nový supermarket'),
 Document(metadata={'_id': '509cfbb35167469fb6d1569817e4eeb5', '_collection_name': 'example_collection'}, page_content='přes oves')]

Qdrant narozdíl od Chromy umožňuje vyhledávat hybridním searchem, tj. kombinací sémantického a fulltextového searche. Pro ten fulltext search budeme potřebovat [dodatečný balíček](https://github.com/qdrant/fastembed) a hlavně model - [Qdrant/bm25](https://huggingface.co/Qdrant/bm25). Všimněme si, že do *create_collection* přibyla i sparse část zodpovědná právě za fulltext search (a teda i dense/sparse klíče - to aby je poté QdrantVectorStore konstruktor našel). Parametr *on_disk* by měl pouze - pokud jsme vše správně pochopil - říkat, zda se předpřipravené sparse vektory BM25 modelu načtou do RAMky anebo zda se budou v případě potřeby načítat z harddisku.

In [5]:
from qdrant_client.models import Distance, VectorParams, SparseVectorParams
from qdrant_client import QdrantClient, models
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode


embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY")
)
sparse_embedding = FastEmbedSparse(model="Qdrant/bm25")

client = QdrantClient(":memory:")
vector_size = len(embeddings_model.embed_query("sample text"))

if not client.collection_exists("example_collection"):
    client.create_collection(
        collection_name="example_collection",
        vectors_config={"dense": VectorParams(size=vector_size, distance=Distance.COSINE)},
        sparse_vectors_config={
            "sparse": SparseVectorParams(index=models.SparseIndexParams(on_disk=False))
        }
    )
    
qdrant_vector_store = QdrantVectorStore(
    client=client,
    collection_name="example_collection",
    embedding=embeddings_model,
    sparse_embedding=sparse_embedding,
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name="dense",
    sparse_vector_name="sparse"
)

In [6]:
str_list = [
    "Skákal pes",
    "přes oves",
    "le chien sautait",
    "a dog was jumping",
    "pes je čtyřnohá šelma",
    "králík rád skáče",
    "byl postaven nový supermarket",
    "ve filmu Vesničko má středisková hráli Labuda a Šafránková",
    "ve filmu Vesničko má středisková hráli Labuda a Šebestová"
]

doc_list = [Document(page_content=one_str) for one_str in str_list]
qdrant_vector_store.add_documents(documents=doc_list)

['0cd726858dc445fb91376da2cb5e93ee',
 '3e0bb91e8b804682a0c7c6fd566dfea2',
 '1452a06ee32741b0a4076e9e502717ff',
 '657a19bcff8c4c4ba4b9fbe365944369',
 '811e64e1fa03457eb5298eb7eb4890b4',
 'a9f8c6668b594b8db95fa9d23a7780cd',
 '7b984b2ca723494e8a624b40e9eddbb8',
 '8a2c4aabaf084ad59f1c6bbacc250afc',
 'ca468e173e1c492e97ce9847fcbc1c73']

In [7]:
user_query = "chci něco, kde bude Šafránková"
similar_docs = qdrant_vector_store.similarity_search_with_score(query=user_query)
similar_docs

[(Document(metadata={'_id': '8a2c4aabaf084ad59f1c6bbacc250afc', '_collection_name': 'example_collection'}, page_content='ve filmu Vesničko má středisková hráli Labuda a Šafránková'),
  1.0),
 (Document(metadata={'_id': '0cd726858dc445fb91376da2cb5e93ee', '_collection_name': 'example_collection'}, page_content='Skákal pes'),
  0.3333333333333333),
 (Document(metadata={'_id': '7b984b2ca723494e8a624b40e9eddbb8', '_collection_name': 'example_collection'}, page_content='byl postaven nový supermarket'),
  0.25),
 (Document(metadata={'_id': 'ca468e173e1c492e97ce9847fcbc1c73', '_collection_name': 'example_collection'}, page_content='ve filmu Vesničko má středisková hráli Labuda a Šebestová'),
  0.2)]

## RAG
Všechny věci výše jsou sice pěkné, ale sami o sobě užitečné nebudou. Hodí se ale, když chceme vytvořit RAG (retrieval augmented generation). Ten nám bude odpovídat na základě [tohoto](https://www.fraternityofshadows.com/Library/ZherisiaGazetteer.pdf) dračákovského spisku - pdfka.

#### Zpracování dokumentu
Abychom mohli dokument zpracovat, musíme ho napřed načíst. Langchain sice nabízí několik možností, jak pdfko nahrát do paměti (např s pomocí unstructured bylíčku), ale instalace zahrnuje hodně balíčků a i pak může celá operace selhat. Proto si v tomto příkladu náš soubor nahrajeme "normálně" bez langchainích utilit a to skrze [pypdf](https://pypi.org/project/pypdf): 

In [5]:
from pypdf import PdfReader

pages_list = []
reader = PdfReader("ZherisiaGazetteer.pdf")
number_of_pages = len(reader.pages)
for one_number_page in range(number_of_pages):
    one_page = reader.pages[one_number_page]
    pages_list.append(one_page.extract_text())
file_content = "\n".join(pages_list)

In [6]:
file_content[0:130]

'Survey on the Zherisian Expedition\n\n\nWriting\nThe Fraternity of Shadows\nJason "Samael" Ambrus\nNicholas "Undead Cabbage" Burke \nDavi'

Pro rozdělení velkého mnohostránkového dokumentu na menší části budeme potřebovat utility z balíčku [langchain-text-splitters](https://pypi.org/project/langchain-text-splitters). Konkrétně sáhneme po *RecursiveCharacterTextSplitter*. Do jeho konstruktoru nasypeme jednak velikost textového fragmentu (*chunk_size*), jednak velikost překryvu s dalšími fragmenty (*chunk_overlap*) - oboje se počítá ve znacích. Následně na takto vytvořeném objektu provoláme metodu *split_text*, do které jako parametr vložíme text - obsah celého původního dokumentu.  
Proč má vůbec [*RecursiveCharacterTextSplitter*](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter) v názvu rekurzi? Jde o to, že má tento objekt v sobě list oddělovačů (defaultně \["\n\n", "\n", " ", ""\]), přičemž oddělovače, které jsou dříve, separují větší entity ("\n\n" je na oddělování paragrafů, " " na oddělování slov). Když jsou fragmenty oddělené prioritnějším oddělovačem, super. Pokud ale i poté jsou fragmenty větší než chunk_size, musí se řezání provést znova, tentokrát s následujícím oddělovačem v pořadí.  
Důvod, proč se tohle dělá, tkví ve skutečnosti, že by pak měl při sobě zůstávat text, který k sobě patří. V kontrastu si představme situaci, kdy bychom text řezaly dejme tomu podle mezerníku - výsledek by byla hromada neukončených nebo naopak z ničeho začínajících vět. 

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=100)
text_chunks = text_splitter.split_text(file_content)

Nyní textové fragmenty vložme do vektorové databáze:

In [25]:
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY")
)
sparse_embedding = FastEmbedSparse(model="Qdrant/bm25")

client = QdrantClient(path="./qdrant_zherasia")
vector_size = len(embeddings_model.embed_query("sample text"))

if not client.collection_exists("zherasia_collection"):
    client.create_collection(
        collection_name="zherasia_collection",
        vectors_config={"dense": VectorParams(size=vector_size, distance=Distance.COSINE)},
        sparse_vectors_config={
            "sparse": SparseVectorParams(index=models.SparseIndexParams(on_disk=False))
        }
    )
    
qdrant_vector_store = QdrantVectorStore(
    client=client,
    collection_name="zherasia_collection",
    embedding=embeddings_model,
    sparse_embedding=sparse_embedding,
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name="dense",
    sparse_vector_name="sparse"
)

In [ ]:
doc_list = [Document(page_content=one_str) for one_str in text_chunks]
qdrant_vector_store.add_documents(documents=doc_list)